[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-02.ipynb)

# 2주차 실습: 학습 루프와 텐서 연산

**목표.** 손실을 계산하고 역전파로 가중치를 갱신하는 **학습 루프 한 바퀴**를 직접 돌려 손실이 내려가는 것을 확인한다. 표기(방언 vs 표준어)가 토크나이저에서 다른 비용으로 이어지는 것도 실측한다.

이 노트북은 이후 모든 주차 실습이 재사용하는 **기본 학습 루프**다. 처음부터 끝까지 한 번 돌려서 감을 잡는 것이 이번 주 목표다.


## 0. 준비

아래 셀을 실행해 필요한 라이브러리를 설치한다. Colab 무료 런타임이면 이미 설치돼 있을 수 있지만, 안전하게 한 번 실행한다.


In [1]:
# 필요한 것 설치 (Colab에서 한 번만)
!pip -q install transformers datasets



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


## 1. 먼저 그냥 실행해 보기

아래 셀들을 위에서부터 차례로 실행하세요. 아무것도 고치지 않아도 끝까지 돌아갑니다.


### 1-1. 텐서와 자동미분

2주차 온라인에서 본 **자동미분**의 동작을 숫자로 확인한다. `requires_grad=True` 텐서에 연산을 걸면 `backward()` 호출 시 그래디언트가 채워진다.


In [2]:
import torch

# x 에 대해 loss 가 미분 가능한지 확인한다
x = torch.tensor(3.0, requires_grad=True)
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

loss = (w * x + b - 10) ** 2   # w*x + b 가 10 에 가까워지도록 하는 손실
loss.backward()

print(f"loss = {loss.item():.4f}")
print(f"d(loss)/dw = {w.grad.item():.4f}")
print(f"d(loss)/dx = {x.grad.item():.4f}")
print(f"d(loss)/db = {b.grad.item():.4f}")


loss = 9.0000
d(loss)/dw = -18.0000
d(loss)/dx = -12.0000
d(loss)/db = -6.0000


### 1-2. 학습 루프 한 바퀴

이제 **순전파 - 손실 - 역전파 - 갱신** 순서를 한 바퀴 돌린다. 가중치가 한 번 갱신된 뒤 손실이 어떻게 변하는지 본다.


In [3]:
import torch

# 데이터: x 가 주어졌을 때 3*x + 1 에 가까운 값을 내는 선형 모델을 맞춘다
x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[4.0], [7.0], [10.0], [13.0]])  # 정답: 3*x + 1

w = torch.tensor([[0.5]], requires_grad=True)
b = torch.tensor([[0.0]], requires_grad=True)

def predict(x):
    return x @ w + b

def mean_squared_error(pred, target):
    return ((pred - target) ** 2).mean()

lr = 0.01

for step in range(200):
    pred = predict(x)
    loss = mean_squared_error(pred, y)
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    if step % 40 == 0:
        print(f"step {step:3d}  loss = {loss.item():.6f}")

print(f"학습 후 w = {w.item():.3f}, b = {b.item():.3f}  (정답: w=3, b=1)")


step   0  loss = 60.375000
step  40  loss = 0.002392
step  80  loss = 0.001861
step 120  loss = 0.001464
step 160  loss = 0.001152
학습 후 w = 3.025, b = 0.926  (정답: w=3, b=1)


### 1-3. 한국어 텍스트 분류 모델 로딩

2주차 온라인에서 본 **사전학습 모델 로딩**이다. 작은 한국어 분류 모델을 불러온다. 3주차에서 이 모델에 어댑터(LoRA)를 붙일 때 같은 코드를 쓴다.


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "monologg/koelectra-base-v3-discriminator"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print(f"모델 전체 파라미터 수: {model.num_parameters():,}")


/private/tmp/claude-501/-Users-macbook-Projects-class-notion/b072bf86-94dd-4a28-b2dc-72062b6c1ba0/scratchpad/deepnlp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 39095.24it/s]


[transformers] ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your d

모델 전체 파라미터 수: 112,922,882


## 2. 한 지점만 바꿔 보기

아래 셀의 `# TODO` 로 표시된 **한 곳만** 바꾸고 다시 실행하세요.

> 바꾸기 전 결과를 먼저 확인해 두면 무엇이 달라졌는지 비교할 수 있습니다.


In [5]:
import torch

# x 가 주어졌을 때 3*x + 1 에 가까운 값을 내는 모델 (1-2 와 동일)
x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[4.0], [7.0], [10.0], [13.0]])

w = torch.tensor([[0.5]], requires_grad=True)
b = torch.tensor([[0.0]], requires_grad=True)

def predict(x):
    return x @ w + b

def mean_squared_error(pred, target):
    return ((pred - target) ** 2).mean()

# TODO: 손실 함수를 mean_squared_error 에서 mean_absolute_error 로 바꿔 보세요
#       아래 def 한 줄을 바꾸면 됩니다.
def mean_absolute_error(pred, target):
    return (pred - target).abs().mean()

loss_fn = mean_absolute_error   # TODO 적용: MSE -> MAE 로 바꿔서 실행

lr = 0.01
for step in range(200):
    pred = predict(x)
    loss = loss_fn(pred, y)
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    if step % 40 == 0:
        print(f"step {step:3d}  loss = {loss.item():.6f}")
print(f"학습 후 w = {w.item():.3f}, b = {b.item():.3f}  (정답: w=3, b=1)")


step   0  loss = 7.250000
step  40  loss = 4.350002
step  80  loss = 1.449999
step 120  loss = 0.000005
step 160  loss = 0.000005
학습 후 w = 3.000, b = 1.000  (정답: w=3, b=1)


## 3. 실무: 방언·표준어 표기와 토크나이저 비용

[실무] 같은 뜻을 담는 데 드는 토큰 비용이 **표기에 따라** 달라지는지 실측한다. 사전학습 토크나이저는 표준어 말뭉치로 학습됐으므로, 방언 표기(제주어)는 같은 뜻의 표준어보다 **토큰 수가 더 많아질** 가능성이 크다.


### 3-1. 먼저 샘플로 감 잡기

데이터를 올리기 전에, 방언과 표준어가 토크나이저에서 어떻게 쪼개지는지 작은 예로 확인한다.


In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("monologg/koelectra-base-v3-discriminator")

pairs = [
    ("혼저 옵서예.", "어서 오세요."),        # 방언 / 표준어
    ("마씀", "말씀"),
    ("고맙수다", "고맙습니다"),
]

for dialect, standard in pairs:
    d_tok = tokenizer.encode(dialect, add_special_tokens=False)
    s_tok = tokenizer.encode(standard, add_special_tokens=False)
    print(f"방언  [{dialect}] -> {len(d_tok)} 토큰 : {tokenizer.convert_ids_to_tokens(d_tok)}")
    print(f"표준  [{standard}] -> {len(s_tok)} 토큰 : {tokenizer.convert_ids_to_tokens(s_tok)}")
    print()


방언  [혼저 옵서예.] -> 6 토큰 : ['혼', '##저', '옵', '##서', '##예', '.']
표준  [어서 오세요.] -> 4 토큰 : ['어서', '오세', '##요', '.']

방언  [마씀] -> 2 토큰 : ['마', '##씀']
표준  [말씀] -> 1 토큰 : ['말씀']

방언  [고맙수다] -> 3 토큰 : ['고맙', '##수', '##다']
표준  [고맙습니다] -> 3 토큰 : ['고맙', '##습', '##니다']



### 3-2. AI Hub 「한국어 방언 발화(제주도)」 데이터 올리기

AI Hub(aihub.or.kr)는 로그인이 필요해 Colab에서 바로 받을 수 없다. 아래 순서로 데이터를 준비한다.

1. [AI Hub](https://www.aihub.or.kr) 로그인 → 「한국어 방언 발화(제주도)」 데이터셋을 내려받는다
2. 이 노트북이 실행 중인 Colab 화면 왼쪽 **폴더(파일)** 아이콘을 누른다
3. 내려받은 파일(예: 방언-표준어 대응 텍스트)을 끌어다 놓는다

데이터가 아직 준비되지 않았다면 **이 셀은 건너뛰고** 3-3의 확인 질문으로 넘어가도 된다. 데이터 형식에 맞게 `data` 리스트를 채우면 그대로 이어서 실행된다.


In [7]:
# data 를 (방언, 표준어) 튜플 리스트로 채우세요. 준비 안 됐으면 그대로 두고 실행해도 됩니다.
data = [
    ("혼저 옵서예.", "어서 오세요."),
    ("하르방", "할아버지"),
    ("쉰 살", "쉰 살"),
]

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("monologg/koelectra-base-v3-discriminator")

print("방언/표준어별 토큰 수:")
for dialect, standard in data:
    d_n = len(tokenizer.encode(dialect, add_special_tokens=False))
    s_n = len(tokenizer.encode(standard, add_special_tokens=False))
    print(f"  방언 [{dialect}] {d_n:2d} 토큰 | 표준 [{standard}] {s_n:2d} 토큰 | 차이 {d_n - s_n:+d}")


방언/표준어별 토큰 수:
  방언 [혼저 옵서예.]  6 토큰 | 표준 [어서 오세요.]  4 토큰 | 차이 +2
  방언 [하르방]  2 토큰 | 표준 [할아버지]  1 토큰 | 차이 +1
  방언 [쉰 살]  2 토큰 | 표준 [쉰 살]  2 토큰 | 차이 +0


## 4. 확인 질문

1. 2번에서 손실 함수를 바꿨을 때 학습 곡선(손실 감소 속도·마지막 값)은 어떻게 달라졌나요? 왜 그렇게 됐다고 생각하나요?
2. 학습 루프에서 **역전파(`loss.backward()`)** 와 **갱신(`w -= lr*w.grad`)** 이 각각 무엇을 하는지 한 문장씩 설명하세요.
3. 방언 표기가 표준어보다 토큰 수가 늘어난 예가 있었나요? 같은 뜻을 담는 데 왜 토큰 비용이 달라지는지 토크나이저의 학습 방식과 연결지어 설명하세요.

답은 아래 셀에 글로 적으면 됩니다. 코드가 아니어도 됩니다.


**답**

1. MSE는 오차를 제곱해서 처음엔 손실이 확 줄고(60.4 → 0.0024) 끝에서 조금 남았다(0.0012, w=3.025, b=0.926). MAE는 기울기 크기가 일정해서 같은 속도로 줄다가 120번째쯤 0.000005까지 떨어지고 w=3.000, b=1.000으로 정확히 맞았다.
2. `loss.backward()`는 각 가중치가 손실에 얼마나·어느 방향으로 영향을 줬는지(기울기)를 계산한다. `w -= lr*w.grad`는 그 반대 방향으로 학습률만큼 가중치를 옮긴다.
3. 있었다. "혼저 옵서예."는 6토큰인데 "어서 오세요."는 4토큰이고, "하르방"은 2토큰인데 "할아버지"는 1토큰이다. 토크나이저가 표준어 글로 학습돼서 방언은 통째로 모르는 말이라 잘게 쪼개기 때문이다.


## 5. 제출

1. 상단 메뉴 **파일 > .ipynb 다운로드** 로 이 노트북을 내려받습니다
2. [저장소](https://github.com/halla-ai/deepnlp-2026)의 `assignments/week-02/<내 학번>/` 에 업로드합니다
3. Pull Request를 엽니다

자세한 방법은 강의 사이트의 **과제 제출** 문서에 있습니다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.
